# Load packages and preprocess data.

In [31]:
# Import Required Packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from xgboost import XGBRegressor

ModuleNotFoundError: No module named 'tensorflow'

In [2]:
# Read in data
df = pd.read_excel('Final Project - data.xlsx')

print(df.info())
df.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5411 entries, 0 to 5410
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Product            5411 non-null   object        
 1   ProductGroup       5411 non-null   object        
 2   weekenddate        5411 non-null   datetime64[ns]
 3   POS                5350 non-null   float64       
 4   Price              5340 non-null   float64       
 5   Shipments          4951 non-null   float64       
 6   RetailerInventory  5411 non-null   int64         
 7   PtsofDistrib       5411 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(2)
memory usage: 338.3+ KB
None


,Product,ProductGroup,weekenddate,POS,Price,Shipments,RetailerInventory,PtsofDistrib
0,SKU1,GROUP1,2020-01-04,1055.0,14.99,749.0,20328,2350
1,SKU1,GROUP1,2020-01-11,1003.0,15.09,1058.0,20298,2351


# Train and Evaluate Random Forest Model.

## Build Model

In [3]:
# Filter out 2025 from the dataset
train_df = df[df['weekenddate'] < '2025-01-01'].copy()
test_df = df[(df['weekenddate'] >= '2025-01-01') & (df['weekenddate'] <= '2025-12-31')].copy()

print("Train dates:", train_df['weekenddate'].min(), "to", train_df['weekenddate'].max())
print("Test dates:", test_df['weekenddate'].min(), "to", test_df['weekenddate'].max())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train dates: 2020-01-04 00:00:00 to 2024-12-28 00:00:00
Test dates: 2025-01-04 00:00:00 to 2025-12-27 00:00:00
Train shape: (4475, 8)
Test shape: (936, 8)


In [4]:
# Feature selection
target = 'POS'   

# Create date-based features
for data in [train_df, test_df]:
    data['year'] = data['weekenddate'].dt.year
    data['month'] = data['weekenddate'].dt.month
    data['week'] = data['weekenddate'].dt.isocalendar().week.astype(int)
    data['quarter'] = data['weekenddate'].dt.quarter

# Drop columns that should not go directly into the model
drop_cols = [target, 'weekenddate']

# Drop missing target values
train_df = train_df.dropna(subset=['POS'])
test_df = test_df.dropna(subset=['POS'])

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df[target]

# One-hot encode categorical columns
cat_cols = ['Product', 'ProductGroup']

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align train/test columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (4414, 27)
X_test shape: (936, 27)


In [5]:
# Train a Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=10,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

## Evaluate Model

In [6]:
# Evaluate the model
y_pred = rf.predict(X_test)

# Metrics
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", rmse)
print("R-Squared:", r2)

Mean Squared Error: 728.9203529201407
R-Squared: 0.765563547631616


In [7]:
# Feature Importance
feature_importances = rf.feature_importances_
print("\nFeature Importances:")
for feature, importance in zip(X_train.columns, feature_importances):
    print(f"{feature}: {importance:.4f}")


Feature Importances:
Price: 0.0316
Shipments: 0.2729
RetailerInventory: 0.3373
PtsofDistrib: 0.0428
year: 0.0089
month: 0.0136
week: 0.0444
quarter: 0.0064
Product_SKU10: 0.0006
Product_SKU11: 0.0000
Product_SKU12: 0.0002
Product_SKU13: 0.0444
Product_SKU14: 0.0330
Product_SKU15: 0.0833
Product_SKU16: 0.0110
Product_SKU17: 0.0008
Product_SKU18: 0.0088
Product_SKU2: 0.0015
Product_SKU3: 0.0277
Product_SKU4: 0.0124
Product_SKU5: 0.0002
Product_SKU6: 0.0005
Product_SKU7: 0.0092
Product_SKU8: 0.0002
Product_SKU9: 0.0052
ProductGroup_GROUP2: 0.0016
ProductGroup_GROUP3: 0.0014


## Interpretation
The Random Forest model shows strong predictive performance, with an RMSE of approximately 728.92 and an R-squared value of 0.766. This indicates that the model explains about 76.6% of the variation in demand, suggesting that the selected features capture a large portion of the underlying demand drivers. While some error remains, the model provides a reasonably accurate forecast for practical business use.

The feature importance results show that demand is primarily driven by operational and availability factors. RetailerInventory (0.3373) and Shipments (0.2729) are the most influential variables, indicating that product availability and supply flow are the strongest drivers of observed demand. This suggests that when inventory is higher and shipments are more consistent, demand fulfillment increases, which aligns with expectations in retail environments. Time-based features such as week (0.0444), month (0.0136), and quarter (0.0064) have relatively low importance, indicating that seasonality plays a smaller role in explaining demand variation compared to operational factors. Similarly, Price has a relatively low importance (0.0316), suggesting that demand in this dataset is less sensitive to pricing changes than to product availability. Product-level variables show moderate importance for specific SKUs. For example, Product_SKU15 (0.0833), Product_SKU13 (0.0444), and Product_SKU14 (0.0330) stand out, indicating that certain products have consistently different demand patterns. However, many SKU variables have near-zero importance, suggesting limited variation or weak differentiation across those products. Product group variables contribute very little to the model, implying that most of the variation is captured at the SKU level rather than the broader product group level.

Overall, the model indicates that demand is primarily driven by supply-side and inventory factors rather than pricing or broad seasonal patterns. This suggests that improving inventory planning and shipment consistency may have a larger impact on demand fulfillment than pricing adjustments or high-level product grouping.

## Hyperparameter tuning and re-evaluation.
Optimize validation using TimeSeriesSplit

Grid search for optimal parameters

In [8]:
# Define model
rf = RandomForestRegressor(random_state=10, n_jobs=-1)

# Define parameter grid
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [6, 8, 12],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Time series split (VERY important for forecasting)
tscv = TimeSeriesSplit(n_splits=5)

# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    n_jobs=-1
)

# Fit
grid_search.fit(X_train, y_train)

# Best results
print("Best Parameters:", grid_search.best_params_)
print("Best RMSE:", -grid_search.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters: {'max_depth': 6, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
Best RMSE: 1732.35062730852


In [9]:
# Train a Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=10,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [10]:
best_rf = grid_search.best_estimator_

# Predict on 2025
y_pred = best_rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print("Tuned RF Results")
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 3))
print("MAE:", round(mae, 2))
print("MAPE:", round(mape, 4))

Tuned RF Results
RMSE: 802.33
R²: 0.716
MAE: 607.51
MAPE: 0.3438


## Interpretation
The tuned Random Forest model shows solid but slightly reduced performance compared to the initial model, with an RMSE of 802.33 and an R² of 0.716. This indicates that the model explains about 71.6% of the variation in demand, which is still a strong result but reflects a modest decline in predictive accuracy. The MAE of 607.51 suggests that, on average, the model’s forecasts deviate from actual demand by about 608 units.

The MAPE of 0.3438 (34.38%) indicates a relatively high average percentage error, meaning that forecasts can deviate substantially from actual values in proportional terms. This suggests that while the model captures overall demand patterns reasonably well, it struggles with precision at the individual SKU-week level, which is common in retail demand forecasting where variability is high.

The decrease in performance compared to the untuned model is likely due to the use of time-series cross-validation during hyperparameter tuning, which provides a more realistic and conservative estimate of model performance by preventing data leakage. As a result, the tuned model is less likely to overfit and may generalize better to future periods, even though its test metrics appear slightly worse.

Overall, the results indicate that the model performs well in capturing general demand trends but still leaves meaningful forecasting error, particularly at a granular level. Further improvements may depend more on feature engineering or refining demand drivers rather than additional hyperparameter tuning.

# Train and Evaluate XGBoost Model

## Build Model

In [11]:
# Make copy of initial dataset
df2 = df.copy()

# Make sure weekenddate is datetime
df2['weekenddate'] = pd.to_datetime(df2['weekenddate'])

# Sort by SKU and date before creating lags
df2 = df2.sort_values(['Product', 'weekenddate'])

# SKU-level lag features
for lag in [1, 2, 3, 6, 12]:
    df2[f'lag_{lag}'] = df2.groupby('Product')['POS'].shift(lag)

# SKU-level rolling features
df2['rolling_mean_3'] = (
    df2.groupby('Product')['POS']
       .transform(lambda x: x.shift(1).rolling(3).mean())
)

df2['rolling_std_3'] = (
    df2.groupby('Product')['POS']
       .transform(lambda x: x.shift(1).rolling(3).std())
)

# Date features
df2['month'] = df2['weekenddate'].dt.month
df2['quarter'] = df2['weekenddate'].dt.quarter
df2['year'] = df2['weekenddate'].dt.year

# Drop rows with missing lag/rolling values
df2 = df2.dropna()

# Set date as index after feature creation
df2 = df2.set_index('weekenddate')

# Convert categorical/object columns to dummy variables
df2 = pd.get_dummies(df2, drop_first=True)

# Train/test split
train_df = df2[df2.index < '2025-01-01'].copy()
test_df = df2[(df2.index >= '2025-01-01') & (df2.index <= '2025-12-31')].copy()

X_train = train_df.drop(columns=['POS'])
y_train = train_df['POS']

X_test = test_df.drop(columns=['POS'])
y_test = test_df['POS']

print("Train dates:", train_df.index.min(), "to", train_df.index.max())
print("Test dates:", test_df.index.min(), "to", test_df.index.max())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Train dates: 2020-03-28 00:00:00 to 2024-12-28 00:00:00
Test dates: 2025-01-04 00:00:00 to 2025-12-27 00:00:00
Train shape: (3897, 34)
Test shape: (837, 34)
X_train shape: (3897, 33)
X_test shape: (837, 33)


In [12]:
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid_1 = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

grid_search_1 = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_1,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_1.fit(X_train, y_train)

best_params_1 = grid_search_1.best_params_
print("Step 1 Best:", best_params_1)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Step 1 Best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}


In [13]:
param_grid_2 = {
    'n_estimators': [best_params_1['n_estimators'] - 100,
                     best_params_1['n_estimators'],
                     best_params_1['n_estimators'] + 100],

    'max_depth': [best_params_1['max_depth'] - 1,
                  best_params_1['max_depth'],
                  best_params_1['max_depth'] + 1],

    'learning_rate': [best_params_1['learning_rate'] * 0.5,
                      best_params_1['learning_rate'],
                      best_params_1['learning_rate'] * 1.5],

    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

grid_search_2 = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_2,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_2.fit(X_train, y_train)

best_model = grid_search_2.best_estimator_
print("Final Best Parameters:", grid_search_2.best_params_)

Fitting 3 folds for each of 108 candidates, totalling 324 fits
Final Best Parameters: {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.7}


## Evaluate Model

In [14]:
# Predictions
y_pred = best_model.predict(X_test)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"R²: {r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop Feature Importances:")
print(feature_importance.head(15).to_string(index=False))

RMSE: 302.8066
MAE: 209.1291
MAPE: 12.58%
R²: 0.9591

Top Feature Importances:
          Feature  Importance
            lag_1    0.578357
   rolling_mean_3    0.095069
          quarter    0.077431
            month    0.022786
    Product_SKU15    0.018383
RetailerInventory    0.015655
            lag_2    0.014619
     Product_SKU3    0.014318
             year    0.013806
           lag_12    0.010761
     PtsofDistrib    0.010379
     Product_SKU9    0.009798
            lag_6    0.009378
    rolling_std_3    0.009374
     Product_SKU2    0.009104


# Train and Evaluate Naive/ARIMA Models

## Build the Model

In [19]:
# Make a copy of the initial datset.
df3 = df.copy()

df3['weekenddate'] = pd.to_datetime(df3['weekenddate'])

df3 = df3.sort_values('weekenddate')

df3.set_index('weekenddate', inplace=True)

y = df3['POS']

In [20]:
df3 = df3.dropna(subset=['POS', 'Price', 'Shipments', 'RetailerInventory'])

In [21]:
# Train/test split
train_df = df3[df3.index < '2025-01-01'].copy()
test_df = df3[(df3.index >= '2025-01-01') & (df3.index <= '2025-12-31')].copy()

# Last observed POS by SKU from training data
last_pos_by_sku = train_df.groupby('Product')['POS'].last()

# Naive forecast for each test row based on its SKU
naive_forecast = test_df['Product'].map(last_pos_by_sku)

# If any SKU in test was not in train, fill with overall train average
naive_forecast = naive_forecast.fillna(train_df['POS'].mean())

naive_forecast.name = 'naive_forecast'

## Evaluate

In [22]:
naive_rmse = np.sqrt(mean_squared_error(test_df['POS'], naive_forecast))
naive_mae = mean_absolute_error(test_df['POS'], naive_forecast)
naive_mape = np.mean(np.abs((test_df['POS'] - naive_forecast) / test_df['POS'])) * 100
naive_r2 = r2_score(test_df['POS'], naive_forecast)

print(f"Naive RMSE: {naive_rmse:.4f}")
print(f"Naive MAE: {naive_mae:.4f}")
print(f"Naive MAPE: {naive_mape:.2f}%")
print(f"Naive R²: {naive_r2:.4f}")

Naive RMSE: 1265.5099
Naive MAE: 993.0944
Naive MAPE: 51.01%
Naive R²: 0.2848


## Build the Model

In [25]:
# Fresh ARIMA dataframe 

df_arima = df.copy()

df_arima['weekenddate'] = pd.to_datetime(df_arima['weekenddate'])

# Keep only what ARIMA needs
df_arima = df_arima[['weekenddate', 'Product', 'POS']].copy()

# Sort by SKU and date
df_arima = df_arima.sort_values(['Product', 'weekenddate'])

# Train/test split
train_arima = df_arima[df_arima['weekenddate'] < '2025-01-01'].copy()
test_arima = df_arima[
    (df_arima['weekenddate'] >= '2025-01-01') &
    (df_arima['weekenddate'] <= '2025-12-31')
].copy()

print("Train dates:", train_arima['weekenddate'].min(), "to", train_arima['weekenddate'].max())
print("Test dates:", test_arima['weekenddate'].min(), "to", test_arima['weekenddate'].max())
print("Train shape:", train_arima.shape)
print("Test shape:", test_arima.shape)

Train dates: 2020-01-04 00:00:00 to 2024-12-28 00:00:00
Test dates: 2025-01-04 00:00:00 to 2025-12-27 00:00:00
Train shape: (4088, 3)
Test shape: (837, 3)


In [26]:
arima_predictions = []

for product in test_arima['Product'].unique():
    
    train_sku = train_arima[train_arima['Product'] == product].sort_values('weekenddate')
    test_sku = test_arima[test_arima['Product'] == product].sort_values('weekenddate')
    
    y_train_sku = train_sku['POS'].astype(float).values
    
    if len(y_train_sku) < 12:
        forecast_values = np.repeat(y_train_sku.mean(), len(test_sku))
    else:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                model = ARIMA(y_train_sku, order=(1, 1, 1))
                fit = model.fit()
                forecast_values = fit.forecast(steps=len(test_sku))
        
        except:
            forecast_values = np.repeat(y_train_sku[-1], len(test_sku))
    
    temp = test_sku.copy()
    temp['ARIMA_Forecast'] = forecast_values
    
    arima_predictions.append(temp)

arima_results = pd.concat(arima_predictions).sort_values(['Product', 'weekenddate'])

## Evaluate

In [27]:
y_true = arima_results['POS']
y_pred = arima_results['ARIMA_Forecast']

arima_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
arima_mae = mean_absolute_error(y_true, y_pred)
arima_mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
arima_r2 = r2_score(y_true, y_pred)

print(f"ARIMA RMSE: {arima_rmse:.4f}")
print(f"ARIMA MAE: {arima_mae:.4f}")
print(f"ARIMA MAPE: {arima_mape:.2f}%")
print(f"ARIMA R²: {arima_r2:.4f}")

ARIMA RMSE: 1265.5099
ARIMA MAE: 993.0944
ARIMA MAPE: 51.01%
ARIMA R²: 0.2848


# Train and Evaluate GRU Model

## Build the Model

In [ ]:
# ── GRU Model Setup ──────────────────────────────────────────────────────────

import pandas as pd
import numpy as np

# Make copy of initial dataset
df4 = df.copy()

# Make sure weekenddate is datetime
df4['weekenddate'] = pd.to_datetime(df4['weekenddate'])

# Keep relevant columns
df4 = df4[['weekenddate', 'Product', 'POS']].copy()

# Sort by SKU and date
df4 = df4.sort_values(['Product', 'weekenddate'])

# Train/test split
train_gru = df4[df4['weekenddate'] < '2025-01-01'].copy()
test_gru = df4[
    (df4['weekenddate'] >= '2025-01-01') &
    (df4['weekenddate'] <= '2025-12-31')
].copy()

print("Train dates:", train_gru['weekenddate'].min(), "to", train_gru['weekenddate'].max())
print("Test dates:", test_gru['weekenddate'].min(), "to", test_gru['weekenddate'].max())
print("Train shape:", train_gru.shape)
print("Test shape:", test_gru.shape)

In [ ]:
# ── Create GRU sequences by SKU ──────────────────────────────────────────────

sequence_length = 12

X_train_seq = []
y_train_seq = []

X_test_seq = []
y_test_seq = []
test_dates = []
test_products = []

scalers = {}

for product in df4['Product'].unique():
    
    product_data = df4[df4['Product'] == product].sort_values('weekenddate').copy()
    
    train_product = product_data[product_data['weekenddate'] < '2025-01-01'].copy()
    test_product = product_data[
        (product_data['weekenddate'] >= '2025-01-01') &
        (product_data['weekenddate'] <= '2025-12-31')
    ].copy()
    
    if len(train_product) <= sequence_length or len(test_product) == 0:
        continue
    
    # Scale POS using training data only
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_product[['POS']])
    
    scalers[product] = scaler
    
    # Training sequences
    for i in range(sequence_length, len(train_scaled)):
        X_train_seq.append(train_scaled[i-sequence_length:i])
        y_train_seq.append(train_scaled[i, 0])
    
    # Combine train tail + test for test sequences
    combined_product = pd.concat([
        train_product.tail(sequence_length),
        test_product
    ]).copy()
    
    combined_scaled = scaler.transform(combined_product[['POS']])
    
    for i in range(sequence_length, len(combined_scaled)):
        X_test_seq.append(combined_scaled[i-sequence_length:i])
        y_test_seq.append(combined_scaled[i, 0])
        test_dates.append(combined_product.iloc[i]['weekenddate'])
        test_products.append(product)

X_train_seq = np.array(X_train_seq)
y_train_seq = np.array(y_train_seq)

X_test_seq = np.array(X_test_seq)
y_test_seq = np.array(y_test_seq)

print("X_train_seq shape:", X_train_seq.shape)
print("y_train_seq shape:", y_train_seq.shape)
print("X_test_seq shape:", X_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

In [ ]:
# ── Build GRU model ──────────────────────────────────────────────────────────

gru_model = Sequential()

gru_model.add(
    GRU(
        units=64,
        activation='tanh',
        return_sequences=True,
        input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])
    )
)

gru_model.add(Dropout(0.2))

gru_model.add(
    GRU(
        units=32,
        activation='tanh'
    )
)

gru_model.add(Dropout(0.2))

gru_model.add(Dense(1))

gru_model.compile(
    optimizer='adam',
    loss='mse'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = gru_model.fit(
    X_train_seq,
    y_train_seq,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## Evaluate

In [ ]:
# ── GRU predictions ──────────────────────────────────────────────────────────

gru_pred_scaled = gru_model.predict(X_test_seq).flatten()

gru_results = pd.DataFrame({
    'weekenddate': test_dates,
    'Product': test_products,
    'Actual_Scaled': y_test_seq,
    'Predicted_Scaled': gru_pred_scaled
})

# Convert scaled predictions back to original POS values by SKU
actual_values = []
predicted_values = []

for product in gru_results['Product'].unique():
    
    mask = gru_results['Product'] == product
    scaler = scalers[product]
    
    actual_scaled = gru_results.loc[mask, 'Actual_Scaled'].values.reshape(-1, 1)
    pred_scaled = gru_results.loc[mask, 'Predicted_Scaled'].values.reshape(-1, 1)
    
    actual_inverse = scaler.inverse_transform(actual_scaled).flatten()
    pred_inverse = scaler.inverse_transform(pred_scaled).flatten()
    
    actual_values.extend(actual_inverse)
    predicted_values.extend(pred_inverse)

gru_results['Actual_POS'] = actual_values
gru_results['Predicted_POS'] = predicted_values

gru_results.head()

In [ ]:
# ── Evaluate GRU model ───────────────────────────────────────────────────────

y_true = gru_results['Actual_POS']
y_pred = gru_results['Predicted_POS']

gru_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
gru_mae = mean_absolute_error(y_true, y_pred)
gru_mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
gru_r2 = r2_score(y_true, y_pred)

print(f"GRU RMSE: {gru_rmse:.4f}")
print(f"GRU MAE: {gru_mae:.4f}")
print(f"GRU MAPE: {gru_mape:.2f}%")
print(f"GRU R²: {gru_r2:.4f}")